# Biohub - Cell Tracking S1.5 Pipeline

このノートブックは、**S1.5フェーズ (スコアアップ戦略)** を実行するための統合検証ノートブックです。

## プロジェクト構成

```text
. (プロジェクトフォルダ)
├── working/                                   # 作業ディレクトリ
│   ├── s1_05_stardist_btrack_pipeline.ipynb   # main処理ノートブック
│   └── kaggle_cell_tracking_competition/      # 【準備1】主催者の公式リポジトリ
│       ├── README.md
│       ├── pyproject.toml
│       ├── tests/
│       ├── scripts/
│       └── src/
│           └── tracking_cellmot/              # 公式の評価用ライブラリ
└── input/                                     # 【準備2】inputデータ
    ├── test/                                  # 提出用データセット (.zarr / .geff)
    │   ├── xxxx.zarr/
    │   └── xxxx.geff/
    └── train/                                 # 訓練用データセット (.zarr / .geff)
        ├── xxxx.zarr/
        └── xxxx.geff/
```

### 【準備1】主催者の公式リポジトリの準備方法
working/配下でclone実行
```batch
git clone https://github.com/royerlab/kaggle_cell_tracking_competition.git
```

### 【準備2】inputデータの準備方法
`./input/` フォルダ配下に展開します。

** データの入手先:**
* [Biohub - Cell Tracking During Development Data](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development/data)で、`Download All` ボタンを押し、ZIPファイルをダウンロード → 解答して展開。

In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

# === オフライン環境でのライブラリ自動インストール ===
import sys
import subprocess

def is_installed(package_name):
    try:
        __import__(package_name)
        return True
    except ImportError:
        return False

def run_pip(cmd_args):
    try:
        from IPython import get_ipython
        ipython = get_ipython()
        if ipython is not None:
            ipython.system(f"pip {cmd_args}")
            return
    except ImportError:
        pass
    subprocess.run([sys.executable, "-m", "pip"] + cmd_args.split(), check=True)

# 1. Zarr パッケージのインストール
if not is_installed("zarr"):
    print("Installing zarr...")
    run_pip("install --no-index --find-links=/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels zarr")
else:
    print("zarr is already installed.")

# 2. tracksdata 関連パッケージのインストール
if not is_installed("tracksdata") or not is_installed("geff") or not is_installed("polars"):
    print("Installing tracksdata and dependencies...")
    run_pip("install --no-index --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-offline-installation-wheels rustworkx bidict ilpy imagecodecs polars")
    run_pip("install --no-index --no-deps --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-offline-installation-wheels geff geff-spec")
    run_pip("install --no-index --no-deps --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-offline-installation-wheels tracksdata")
else:
    print("tracksdata and dependencies are already installed.")

print("Offline installation steps completed.")

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Offline installation check completed. (Elapsed: {_cell_elapsed:.2f}s)")


In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

import os
import sys
import glob
import time
import numpy as np
import pandas as pd
from skimage.feature import blob_dog

# === Polarsのエラー回避モンキーパッチ ===
import polars as pl
if not hasattr(pl, 'Float16'):
    pl.Float16 = pl.Float32

# === プロジェクト構成 ===
# 詳細な構成については、ノートブック先頭（最初のセル）を参照してください。

# === パス設定 (静的解決) ===
# working/ フォルダ内から実行するため、同一フォルダにある公式リポジトリのパスをそのまま登録します
target_path = os.path.abspath(os.path.join(os.getcwd(), 'kaggle_cell_tracking_competition', 'src'))

if os.path.exists(os.path.join(target_path, 'tracking_cellmot')):
    if target_path not in sys.path:
        sys.path.insert(0, target_path)
    print(f"Path set successfully: {target_path}")
else:
    # フォールバック
    fallback_path = os.path.abspath(os.path.join(os.getcwd(), 'src'))
    if fallback_path not in sys.path:
        sys.path.insert(0, fallback_path)
    print(f"Warning: Expected path not found. Using fallback path: {fallback_path}")

import tracksdata as td
from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import node_recall
from tracksdata.metrics import DistanceMatching
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Imports and path resolution completed. (Elapsed: {_cell_elapsed:.2f}s)")


## 【ステップ 1 & 2】検出評価と検出器の向上

In [ ]:
def detect_nodes_baseline(dataset_path, min_sigma=2.0, max_sigma=5.0, threshold=0.05, max_frames=None):
    """
    ベースライン検出器 (blob_dog)
    """
    ds = open_dataset(dataset_path, normalize=True, require_tracks=False, device='cpu')
    images = ds.image
    n_frames = images.shape[0]
    from tqdm import tqdm
    if max_frames is not None:
        n_frames = min(n_frames, max_frames)
    
    nodes = []
    global_node_id = 0
    
    for t in tqdm(range(n_frames), desc="Detecting cells in 3D frames"):
        frame = images[t]
        if hasattr(frame, 'numpy'):
            frame = frame.numpy()
        
        img_min, img_max = frame.min(), frame.max()
        if img_max > img_min:
            img_norm = (frame.astype(np.float32) - img_min) / (img_max - img_min)
        else:
            img_norm = np.zeros_like(frame, dtype=np.float32)
            
        blobs = blob_dog(img_norm, min_sigma=min_sigma, max_sigma=max_sigma, threshold=threshold)
        
        for blob in blobs:
            z, y, x, r = blob
            nodes.append({
                'node_id': global_node_id,
                't': t,
                'z': float(z),
                'y': float(y),
                'x': float(x)
            })
            global_node_id += 1
            
    return pd.DataFrame(nodes)


def evaluate_detection_only(pred_nodes_df, gt_graph, scale, max_distance=7.0):
    """
    ステップ1: 検出単体の評価 (Recallの測定)
    """
    pred_graph = td.graph.InMemoryGraph()
    for key in ('z', 'y', 'x'):
        pred_graph.add_node_attr_key(key, pl.Float64, 0.0)
        
    for row in pred_nodes_df.itertuples():
        pred_graph.add_node({
            't': int(row.t),
            'z': float(row.z),
            'y': float(row.y),
            'x': float(row.x)
        })
        
    matching = DistanceMatching(max_distance=max_distance, scale=scale)
    pred_graph.match(gt_graph, matching=matching)
    
    recall = node_recall(pred_graph, gt_graph)
    
    return {
        'node_recall': recall,
        'num_pred_nodes': pred_graph.num_nodes(),
        'num_gt_nodes': gt_graph.num_nodes()
    }

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Cell Cell detection and evaluation functions defined. (Elapsed: {_cell_elapsed:.2f}s)")



### 1.1 検出結果の3D視覚化 (Error Analysis)

予測された細胞位置 (ノード) と GT を3D空間にプロットし、公式マッチング基準 (7 µm) で正しく検出できたもの (TP: 緑)、余分な検出 (FP: 赤)、見落とした正解 (FN: 青) を色分けして可視化します。

In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import KDTree

def plot_detection_dashboard(image_3d, pred_nodes_df, gt_graph, t_selected=0, max_distance=7.0, scale=[1.0, 1.0, 1.0]):
    """
    指定したフレームにおける検出結果を Plotly で 3D プロット（MIP重ね合わせ付き）します。
    左側：3D空間での TP/FP/FN 点群（マウス操作可能）
    右側：MIP画像のサーフェス上に投影した点群
    """
    scale = np.array(scale)
    nz, ny, nx = image_3d.shape
    mip = np.max(image_3d, axis=0)
    # 物理座標系に合わせるため、MIPのグリッドに物理スケールを適用
    x_grid, y_grid = np.meshgrid(np.arange(nx) * scale[2], np.arange(ny) * scale[1])
    # クリッピングを防ぐため、MIP画像(Surface)のZ位置をわずかに下げる
    z_surface = np.full_like(mip, -0.05)

    # GTの座標抽出
    gt_nodes_pl = gt_graph.node_attrs(attr_keys=['t', 'z', 'y', 'x'])
    gt_nodes_df = gt_nodes_pl.to_pandas()
    
    p_nodes = pred_nodes_df[pred_nodes_df['t'] == t_selected].copy()
    g_nodes = gt_nodes_df[gt_nodes_df['t'] == t_selected].copy()
    
    p_coords = p_nodes[['z', 'y', 'x']].values * scale
    # GT座標もピクセル座標なので、物理座標系に合わせる(scaleを掛ける)
    g_coords = g_nodes[['z', 'y', 'x']].values * scale
    
    tp_p_idx, tp_g_idx = [], []
    if len(p_coords) > 0 and len(g_coords) > 0:
        tree = KDTree(g_coords)
        distances, indices = tree.query(p_coords, distance_upper_bound=max_distance)
        
        matched_g = set()
        for p_idx, (d, g_idx) in enumerate(zip(distances, indices)):
            if d <= max_distance and g_idx not in matched_g:
                tp_p_idx.append(p_idx)
                tp_g_idx.append(g_idx)
                matched_g.add(g_idx)
                
    tp_coords = p_coords[tp_p_idx] if tp_p_idx else np.empty((0, 3))
    fp_coords = np.delete(p_coords, tp_p_idx, axis=0) if len(p_coords) > 0 else np.empty((0, 3))
    fn_coords = np.delete(g_coords, tp_g_idx, axis=0) if len(g_coords) > 0 else np.empty((0, 3))
    
    # サブプロットの作成 (両方とも 3D シーン)
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}]],
        subplot_titles=(
            "1. 3D Point Cloud (Interactive TP/FP/FN)",
            "2. 3D MIP Surface Overlay (Flat Z=0.0 Front Layer)"
        )
    )
    
    # ------------------------------------------
    # 左側: 3D Point Cloud
    # ------------------------------------------
    if len(tp_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=tp_coords[:, 2], y=tp_coords[:, 1], z=tp_coords[:, 0],
            mode='markers', name=f'TP (Correct): {len(tp_coords)}',
            marker=dict(size=4, color='lime', opacity=0.8, symbol='circle')
        ), row=1, col=1)
    if len(fp_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=fp_coords[:, 2], y=fp_coords[:, 1], z=fp_coords[:, 0],
            mode='markers', name=f'FP (Extra): {len(fp_coords)}',
            marker=dict(size=4, color='red', opacity=0.8, symbol='x')
        ), row=1, col=1)
    if len(fn_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=fn_coords[:, 2], y=fn_coords[:, 1], z=fn_coords[:, 0],
            mode='markers', name=f'FN (Missing): {len(fn_coords)}',
            marker=dict(size=4, color='cyan', opacity=0.8, symbol='diamond')
        ), row=1, col=1)
        
    # ------------------------------------------
    # 右側: 3D MIP Surface Overlay
    # ------------------------------------------
    # 1. MIPサーフェス画像 (Z=-0.05平面)
    fig.add_trace(go.Surface(
        x=x_grid, y=y_grid, z=z_surface,
        surfacecolor=mip, colorscale='Gray', showscale=False,
        hoverinfo='x+y+z', name='MIP'
    ), row=1, col=2)
    
    # 2. 検出点の投影 (Z=0 平面。透視投影によるズレを防ぐためMIPのほぼ真上に配置)
    proj_z = 0.0
    if len(tp_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=tp_coords[:, 2], y=tp_coords[:, 1], z=np.full(len(tp_coords), proj_z),
            mode='markers', name='TP (Projected)',
            marker=dict(size=4, color='lime', opacity=0.9, symbol='circle'),
            showlegend=False
        ), row=1, col=2)
    if len(fp_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=fp_coords[:, 2], y=fp_coords[:, 1], z=np.full(len(fp_coords), proj_z),
            mode='markers', name='FP (Projected)',
            marker=dict(size=4, color='red', opacity=0.9, symbol='x'),
            showlegend=False
        ), row=1, col=2)
    if len(fn_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=fn_coords[:, 2], y=fn_coords[:, 1], z=np.full(len(fn_coords), proj_z),
            mode='markers', name='FN (Projected)',
            marker=dict(size=4, color='cyan', opacity=0.9, symbol='diamond'),
            showlegend=False
        ), row=1, col=2)
        
    fig.update_layout(
        title_text=f"Cell Detection Dual 3D Interactive Dashboard (Frame {t_selected})",
        scene1=dict(
            xaxis=dict(title='X (Width)'),
            yaxis=dict(title='Y (Height)'),
            zaxis=dict(title='Z (Depth)', range=[0, nz * scale[0]]),
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=0.6),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
        ),
        scene2=dict(
            xaxis=dict(title='X (Width)'),
            yaxis=dict(title='Y (Height)'),
            zaxis=dict(title='Z (Depth)', range=[-0.5, 0.5]),
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=0.1),
            camera=dict(eye=dict(x=0, y=0, z=2.2)) # 真上から見下ろす視点
        ),
        height=650,
        margin=dict(l=20, r=20, b=20, t=60)
    )
    fig.show()

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Interactive Plotly 3D visualization dashboard functions defined. (Elapsed: {_cell_elapsed:.2f}s)")


## 【ステップ 3 & 4】トラッキング評価とトラッカーの向上

In [ ]:
# # === 実行時間計測の開始 ===
# import time
# _cell_start_time = time.time()
# 
# # # def evaluate_tracking_only(pred_edges, gt_graph, gt_nodes_df, scale):
# # #     """
# # #     ステップ3: トラッキング単体の評価 (GTノードを直接入力)
# # #     """
# # #     from tracking_cellmot.metrics import evaluate
# # #     
# # #     pred_graph = td.graph.InMemoryGraph()
# # #     for key in ('z', 'y', 'x'):
# # #         pred_graph.add_node_attr_key(key, pl.Float64, 0.0)
# # #         
# # #     for row in gt_nodes_df.itertuples():
# # #         pred_graph.add_node({
# # #             'node_id': int(row.node_id),
# # #             't': int(row.t),
# # #             'z': float(row.z),
# # #             'y': float(row.y),
# # #             'x': float(row.x)
# # #         })
# # #         
# # #     for edge in pred_edges:
# # #         pred_graph.add_edge(int(edge['source_id']), int(edge['target_id']))
# # #         
# # #     res = evaluate(pred_graph, gt_graph, scale=scale)
# # #     
# # #     edge_denom = res.edge_tp + res.edge_fp + res.edge_fn
# # #     edge_jaccard = res.edge_tp / edge_denom if edge_denom > 0 else 1.0
# # #     
# # #     return {
# # #         'edge_jaccard': edge_jaccard,
# # #         'edge_tp': res.edge_tp,
# # #         'edge_fp': res.edge_fp,
# # #         'edge_fn': res.edge_fn
# # #     }
# # 
# print("Tracking evaluation functions defined (commented out).")
# 
# print("Tracking evaluation functions defined (commented out).")
# 
# # === 実行時間計測の終了 ===
# import datetime
# _cell_elapsed = time.time() - _cell_start_time
# _jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
# _cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
# print(f"\n[{_cell_current_time} JST] Cell execution completed. (Elapsed: {_cell_elapsed:.2f}s)")

# print("Tracking evaluation functions defined (commented out).")


## 【ステップ 5 & 6】統合評価と間引き (Pruning) の実行

検出予測ノードを用いたトラッキングの実行と公式評価（統合評価）、および不要なノードやエッジを削る間引き (Pruning) 処理を行います。

In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

# # 3. 統合評価 (検出予測ノードを用いたトラッキング実行と公式評価)
# print('\n--- Running Complete End-to-End Evaluation ---')
# # complete_edges = run_btrack_tracking(pred_nodes, max_search_radius=25.0)
# # complete_res = evaluate_complete(pred_graph, gt_graph, scale=scale)
# 
# # 4. 間引き (Pruning) / 後処理の実行
# print('\n--- Running Pruning / Post-processing ---')
# # pruned_nodes, pruned_edges = prune_tracks(pred_nodes, complete_edges)
# 
# # print("Complete evaluation and pruning functions processed (commented out).")

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Cell execution completed. (Elapsed: {_cell_elapsed:.2f}s)")


## ベースライン全体の実行とテスト

データセットを読み込み、これまでに実装した検出器の検証を行います。

In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

# データ読み込みとmain()テスト実行
import sys
    DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'input', 'train'))
dataset_paths = glob.glob(os.path.join(DATA_DIR, '*.zarr')) + glob.glob(os.path.join(DATA_DIR, '*.geff'))
if not dataset_paths:
    dataset_paths = glob.glob('/kaggle/input/**/train/*.zarr', recursive=True) + glob.glob('/kaggle/input/**/train/*.geff', recursive=True)

if not dataset_paths:
    raise FileNotFoundError("Error: No .zarr or .geff datasets found in the data directories.")

target_dataset_path = dataset_paths[0]
print(f'Target dataset: {target_dataset_path}')

# GTのロード (失敗時はエラー終了)
try:
    ds_gt = open_dataset(target_dataset_path, normalize=True, require_tracks=True, device='cpu')
except Exception as e:
    print(f"Error: Failed to load dataset {target_dataset_path}. Reason: {e}", file=sys.stderr)
    raise e

gt_graph = ds_gt.tracks
scale = ds_gt.scale

# 1. 検出評価
print('\n--- Running Node Detection Evaluation ---')
pred_nodes = detect_nodes_baseline(target_dataset_path, max_frames=1)  # デバッグ用に最初の1フレームのみ実行
det_res = evaluate_detection_only(pred_nodes, gt_graph, scale=scale)
print(f'Node Recall: {det_res["node_recall"]:.4f}')

# 可視化の実行 (MIP画像の取得とダッシュボード表示)
t_selected = 0  # 可視化対象の時間フレームを選択
print(f'\n--- Visualizing frame {t_selected} cell centroids ---')
img_3d = ds_gt.image[t_selected]
if hasattr(img_3d, 'numpy'):
    img_3d = img_3d.numpy()
    plot_detection_dashboard(img_3d, pred_nodes, gt_graph, t_selected=t_selected, scale=scale)

# 2. トラッキング評価 (GTノードを入力) -> コメントアウト
# print('\n--- Running Tracking Evaluation with GT Nodes ---')
# from track.btrack_tracker import run_btrack_tracking
# gt_nodes_pl = gt_graph.node_attrs(attr_keys=['node_id', 't', 'z', 'y', 'x'])
# gt_nodes_df = gt_nodes_pl.to_pandas()
# 
# # btrack実行
# edges = run_btrack_tracking(gt_nodes_df, max_search_radius=25.0)
# track_res = evaluate_tracking_only(edges, gt_graph, gt_nodes_df, scale=scale)
# print(f'Edge Jaccard: {track_res["edge_jaccard"]:.4f} (TP={track_res["edge_tp"]}, FP={track_res["edge_fp"]}, FN={track_res["edge_fn"]})')

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Pipeline detection evaluation completed. (Elapsed: {_cell_elapsed:.2f}s)")
